# Wan 2.2 (5B) di Kaggle: video AI natural, gratis, tanpa watermark

Notebook ini membuat klip video dari **teks** atau **foto produk** dengan model open-source
[Wan2.2-TI2V-5B](https://huggingface.co/Wan-AI/Wan2.2-TI2V-5B-Diffusers) (Apache 2.0, boleh komersial).
Hasil akhirnya **1080×1920, 30 fps**, sama dengan proyek KanvasCut, jadi tinggal diimpor.

**Yang membuat hasil lebih natural:**
- Preset `natural` memakai **720p**, resolusi asli model (480p hanya untuk draf cepat).
- `prompt_natural()` menyusun prompt realistis: gaya `hp` (video HP), `studio`, atau `lifestyle`.
- Negative prompt ditambah supaya tidak terlihat "AI banget" (kartun, kulit plastik, morphing, kedip).
- Finishing otomatis: gerakan dihaluskan (24 → 30 fps dengan interpolasi gerak) dan gambar diperbesar ke 1080p.
- **Foto asli → video** biasanya paling natural, karena berangkat dari gambar nyata.

**Dua GPU sekaligus:** dengan *GPU T4 x2*, kedua GPU membuat video bersamaan saat batch, sekitar 2× lebih banyak klip.

## Sebelum mulai (panel kanan → *Settings*)
1. **Accelerator: GPU T4 x2**. Satu GPU juga bisa, tapi lebih lambat.
2. **Internet: On**. Ini butuh akun Kaggle yang sudah verifikasi nomor HP.
3. Jalankan sel dari atas ke bawah. Unduh dan muat model butuh sekitar 10–20 menit, sekali per sesi.

## Preset
Waktunya belum diukur langsung. Lihat angka detik per step (`s/it`) di progress bar saat tes.

| Preset | Ukuran | Durasi | Kegunaan |
|---|---|---|---|
| `tes` | 480×832 | ~1 dtk | cek mesin jalan |
| `cepat` | 480×832 | ~2 dtk | draf, coba prompt |
| `natural` | 704×1280 | ~2 dtk | **klip final (disarankan)** |
| `natural_panjang` | 704×1280 | ~3,4 dtk | klip final lebih panjang, ~2× lebih lama |
| `hd` | 704×1280 | ~5 dtk | sangat lama, berisiko kehabisan memori |

Kuota GPU Kaggle terbatas per minggu (cek di profil). Coba prompt dengan `cepat`, lalu buat versi final dengan `natural`.

**Ingat:** klip AI wajib diberi label *AI-generated* di TikTok, dan produk harus tampil akurat.

## 1. Pasang library

In [ ]:
!pip install -q -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg

## 2. Fungsi pembuat video

In [ ]:
import os
os.environ["HF_HOME"] = "/tmp/hf"                                   # /kaggle/working cuma 20 GB
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # kurangi fragmentasi memori GPU

In [ ]:
import copy, gc, json, queue, random, subprocess, threading, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from types import SimpleNamespace

import torch
from PIL import Image, ImageOps
from diffusers import WanPipeline, WanImageToVideoPipeline
from diffusers.utils import export_to_video

# Negative prompt bawaan Wan (bahasa Mandarin) + tambahan untuk menekan tampilan "AI banget".
NEGATIVE_PROMPT = (
    "色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，最差质量，低质量，"
    "JPEG压缩残留，丑陋的，残缺的，多余的手指，画得不好的手部，画得不好的脸部，畸形的，毁容的，形态畸形的肢体，"
    "手指融合，静止不动的画面，杂乱的背景，三条腿，背景人很多，倒着走, "
    "cartoon, anime, CGI, 3D render, illustration, plastic look, waxy skin, oversaturated, overexposed, "
    "flickering, jitter, morphing, warping, distorted text, garbled logo, watermark, subtitles, blurry, low quality"
)

# Ukuran kelipatan 32, frame = 4k+1, model dilatih 24 fps di 720p (704x1280).
PRESETS = {
    "tes":             dict(height=832,  width=480, frames=25,  steps=20, cfg=5.0),  # ~1 dtk, cek mesin
    "cepat":           dict(height=832,  width=480, frames=49,  steps=25, cfg=5.0),  # ~2 dtk, draf
    "natural":         dict(height=1280, width=704, frames=49,  steps=40, cfg=5.0),  # ~2 dtk 720p, kualitas
    "natural_panjang": dict(height=1280, width=704, frames=81,  steps=40, cfg=5.0),  # ~3,4 dtk 720p
    "hd":              dict(height=1280, width=704, frames=121, steps=40, cfg=5.0),  # ~5 dtk 720p, sangat lama
}
FPS = 24

GAYA = {
    "hp": ("Realistic smartphone video, handheld camera with subtle natural shake, natural daylight, "
           "true-to-life colors, real textures, authentic everyday setting"),
    "studio": ("Professional commercial product video, soft diffused studio lighting, clean seamless background, "
               "crisp details, smooth slow camera motion"),
    "lifestyle": ("Cinematic lifestyle footage, warm natural light, shallow depth of field, gentle slow camera "
                  "movement, realistic people and environment, subtle film grain"),
}


def prompt_natural(subjek, gerakan="", suasana="", kamera="", gaya="hp"):
    """Susun prompt realistis. Isi dalam bahasa Inggris, contoh:
    prompt_natural("a woman's hand holding a pink lip tint", "she twists the cap open",
                   "bright bathroom vanity in the morning", "slow push-in close-up")
    """
    bagian = [subjek, gerakan, suasana, kamera, GAYA[gaya],
              "photorealistic, natural physics, smooth natural motion, highly detailed"]
    return ". ".join(b.strip().rstrip(".") for b in bagian if b and b.strip()) + "."


def make_worker(tokenizer, transformer, vae, scheduler):
    """Satu worker = satu GPU (transformer + VAE sendiri, scheduler sendiri)."""
    common = dict(tokenizer=tokenizer, text_encoder=None, vae=vae, scheduler=copy.deepcopy(scheduler),
                  transformer=transformer, expand_timesteps=True)
    w = SimpleNamespace(t2v=WanPipeline(**common), i2v=WanImageToVideoPipeline(**common))
    w.device = w.t2v._execution_device
    return w


def make_encoder(tokenizer, text_encoder, scheduler):
    return WanPipeline(tokenizer=tokenizer, text_encoder=text_encoder, vae=None,
                       scheduler=scheduler, transformer=None)


def _ffmpeg():
    import imageio_ffmpeg
    return imageio_ffmpeg.get_ffmpeg_exe()


def finishing(src, dst, halus=True, upscale=True):
    """Haluskan gerakan (interpolasi 24 -> 30 fps) dan perbesar ke 1080x1920, sama dengan proyek KanvasCut."""
    vf = []
    if halus:
        vf.append("minterpolate=fps=30:mi_mode=mci:mc_mode=aobmc:me_mode=bidir:vsbmc=1")
    if upscale:
        vf += ["scale=1080:1920:force_original_aspect_ratio=increase:flags=lanczos",
               "crop=1080:1920", "unsharp=5:5:0.4:5:5:0.0"]
    if not vf:
        return src
    cmd = [_ffmpeg(), "-y", "-loglevel", "error", "-i", str(src), "-vf", ",".join(vf),
           "-c:v", "libx264", "-preset", "medium", "-crf", "17", "-pix_fmt", "yuv420p", str(dst)]
    subprocess.run(cmd, check=True)
    return dst


class WanStudio:
    def __init__(self, encoder, workers, te_device, out_dir, finishing=True):
        self.enc = encoder
        self.workers = list(workers)
        self.te_device = torch.device(te_device)
        self.out = Path(out_dir)
        (self.out / "mentah").mkdir(parents=True, exist_ok=True)
        self.finishing = finishing
        self._cache, self._lock = {}, threading.Lock()
        self._post = ThreadPoolExecutor(max_workers=2)   # ffmpeg jalan di CPU, GPU tidak menunggu

    # ---------- teks -> embedding ----------
    @torch.no_grad()
    def _encode_on(self, text, device):
        emb, _ = self.enc.encode_prompt(prompt=text, do_classifier_free_guidance=False,
                                        max_sequence_length=512, device=device)
        return emb

    def text_encoder_to_cpu(self):
        if self.te_device.type != "cpu":
            self.enc.text_encoder.to("cpu")
            self.te_device = torch.device("cpu")
            gc.collect(); torch.cuda.empty_cache()

    def encode(self, text):
        with self._lock:
            if text not in self._cache:
                if self.te_device.type == "cpu":
                    print("  (encode prompt di CPU, agak lama...)")
                try:
                    emb = self._encode_on(text, self.te_device)
                    ok = bool(torch.isfinite(emb).all())
                except RuntimeError as e:  # mis. bf16 tidak didukung GPU ini
                    print("Text encoder gagal di", self.te_device, "->", e)
                    ok = False
                if not ok:
                    if self.te_device.type == "cpu":
                        raise RuntimeError("Text encoder menghasilkan NaN di CPU.")
                    print("Pindah text encoder ke CPU (lebih lambat, tapi aman).")
                    self.text_encoder_to_cpu()
                    emb = self._encode_on(text, self.te_device)
                self._cache[text] = emb.cpu()
            return self._cache[text]

    def encode_semua(self, jobs):
        teks = {NEGATIVE_PROMPT} | {self._job(j)["prompt"] for j in jobs}
        baru = [t for t in teks if t not in self._cache]
        for i, t in enumerate(baru, 1):
            print(f"encode prompt {i}/{len(baru)}")
            self.encode(t)

    # ---------- GPU kedua ----------
    def tambah_worker(self, worker):
        self.workers.append(worker)
        print(f"Worker aktif: {[str(w.device) for w in self.workers]}")

    # ---------- pembuatan video ----------
    @staticmethod
    def _job(job):
        return {"prompt": job} if isinstance(job, str) else dict(job)

    def _nama(self, seed):
        with self._lock:
            n = len(list((self.out / "mentah").glob("*.mp4"))) + 1
            name = f"wan_{time.strftime('%Y%m%d')}_{n:03d}_{seed}"
            (self.out / "mentah" / f"{name}.mp4").touch()   # pesan nama, aman untuk 2 thread
            return name

    def _render(self, w, prompt, image, preset, seed, negative_prompt):
        p = PRESETS[preset]
        g = torch.Generator(device=w.device).manual_seed(seed)
        kw = dict(prompt_embeds=self.encode(prompt).to(w.device),
                  negative_prompt_embeds=self.encode(negative_prompt).to(w.device),
                  height=p["height"], width=p["width"], num_frames=p["frames"],
                  num_inference_steps=p["steps"], guidance_scale=p["cfg"], generator=g)
        if image is None:
            return w.t2v(**kw).frames[0]
        img = image if isinstance(image, Image.Image) else Image.open(image)
        img = ImageOps.fit(ImageOps.exif_transpose(img).convert("RGB"), (p["width"], p["height"]), Image.LANCZOS)
        return w.i2v(image=img, **kw).frames[0]

    def generate(self, prompt, image=None, preset="natural", seed=None, name=None,
                 negative_prompt=NEGATIVE_PROMPT, worker=0, tunggu=True):
        w = self.workers[worker] if isinstance(worker, int) else worker
        seed = random.randrange(2**31) if seed is None else int(seed)
        name = name or self._nama(seed)
        t0 = time.time()
        frames = self._render(w, prompt, image, preset, seed, negative_prompt)
        mentah = self.out / "mentah" / f"{name}.mp4"
        export_to_video(frames, str(mentah), fps=FPS)
        detik = round(time.time() - t0, 1)
        del frames
        gc.collect(); torch.cuda.empty_cache()
        info = dict(file=f"{name}.mp4", prompt=prompt, image=str(image) if image else None,
                    preset=preset, seed=seed, gpu=str(w.device), detik=detik)
        print(f"✅ {name}.mp4  {detik} dtk di {w.device}")

        def selesai():
            final = finishing(mentah, self.out / f"{name}.mp4") if self.finishing else mentah
            with self._lock, open(self.out / "log.jsonl", "a") as f:
                f.write(json.dumps(info, ensure_ascii=False) + "\n")
            return Path(final)

        fut = self._post.submit(selesai)
        return fut.result() if tunggu else fut

    def batch(self, jobs, preset="natural"):
        jobs = [self._job(j) for j in jobs]
        self.encode_semua(jobs)
        q = queue.Queue()
        for i, j in enumerate(jobs, 1):
            q.put((i, j))
        hasil, t0 = [], time.time()

        def jalan(w):
            for pipe in (w.t2v, w.i2v):
                pipe.set_progress_bar_config(disable=len(self.workers) > 1)
            while True:
                try:
                    i, job = q.get_nowait()
                except queue.Empty:
                    return
                print(f"[{i}/{len(jobs)}] {w.device}: {job['prompt'][:70]}")
                try:
                    hasil.append(self.generate(preset=job.pop("preset", preset), worker=w, tunggu=False, **job))
                except torch.cuda.OutOfMemoryError:
                    print(f"❌ [{i}] {w.device} kehabisan memori. Coba preset lebih kecil.")
                    gc.collect(); torch.cuda.empty_cache()
                except Exception as e:  # satu klip gagal tidak menghentikan batch
                    print(f"❌ [{i}] gagal: {e}")

        with ThreadPoolExecutor(max_workers=len(self.workers)) as ex:
            list(ex.map(jalan, self.workers))
        files = [f.result() for f in hasil]
        menit = (time.time() - t0) / 60
        print(f"\nSelesai: {len(files)}/{len(jobs)} klip dalam {menit:.1f} menit "
              f"({menit / max(len(files), 1):.1f} menit/klip rata-rata)")
        return files

## 3. Unduh dan muat model
Model dimuat langsung ke GPU supaya RAM tidak penuh.

In [ ]:
import torch
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, UMT5EncoderModel
from diffusers import AutoencoderKLWan, WanTransformer3DModel, UniPCMultistepScheduler

assert torch.cuda.is_available(), "GPU belum aktif: Settings → Accelerator → GPU T4 x2"
n_gpu = torch.cuda.device_count()
for i in range(n_gpu):
    print(f"cuda:{i}", torch.cuda.get_device_name(i))

# Text encoder (UMT5, bf16) di GPU kedua selama encode prompt, lalu pindah ke CPU saat GPU kedua dipakai membuat video.
TE_DEVICE = "cuda:1" if n_gpu >= 2 else "cpu"

MODEL_ID = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
model_dir = snapshot_download(MODEL_ID, ignore_patterns=["assets/*", "examples/*"])

tokenizer = AutoTokenizer.from_pretrained(model_dir, subfolder="tokenizer")
scheduler = UniPCMultistepScheduler.from_pretrained(model_dir, subfolder="scheduler")
text_encoder = UMT5EncoderModel.from_pretrained(
    model_dir, subfolder="text_encoder", torch_dtype=torch.bfloat16, device_map=TE_DEVICE).eval()


def muat_worker(device):
    # T4/P100 tidak punya bf16 native, jadi transformer pakai fp16. VAE fp32 supaya warna bersih.
    tr = WanTransformer3DModel.from_pretrained(
        model_dir, subfolder="transformer", torch_dtype=torch.float16, device_map=device)
    vae = AutoencoderKLWan.from_pretrained(
        model_dir, subfolder="vae", torch_dtype=torch.float32, device_map=device)
    vae.enable_tiling()   # decode per potongan supaya memori tidak melonjak di akhir
    gc.collect()
    return make_worker(tokenizer, tr, vae, scheduler)


studio = WanStudio(make_encoder(tokenizer, text_encoder, scheduler), [muat_worker("cuda:0")],
                   TE_DEVICE, out_dir="/kaggle/working/wan_output", finishing=True)
for i in range(n_gpu):
    print(f"cuda:{i} terpakai {torch.cuda.memory_allocated(i)/2**30:.1f} GB")

## 4. Tes pertama
Jalankan dengan `preset="tes"` dulu. Kalau lancar, ganti ke `"natural"` untuk melihat kualitas aslinya.

In [ ]:
from IPython.display import Video

prompt = prompt_natural(
    subjek="a white ceramic coffee mug on a wooden kitchen table",
    gerakan="steam gently rising from the hot coffee",
    suasana="cozy kitchen, soft morning sunlight through the window",
    kamera="slow push-in, close-up",
    gaya="hp",
)
print(prompt)
hasil = studio.generate(prompt, preset="tes", seed=42)
Video(str(hasil), embed=True, width=270)

## 5. Foto produk → video
1. Upload foto produk sebagai Dataset Kaggle (*Add Input* → *Upload*). Foto ada di `/kaggle/input/<nama-dataset>/`.
2. Isi `FOTO` di bawah.

Prompt sebaiknya menggambarkan **gerakan kamera dan suasana**, bukan mengubah produknya.
Periksa hasilnya: kalau logo atau tulisan di produk berubah bentuk, jangan pakai klip itu.

In [ ]:
import glob
print("\n".join(glob.glob("/kaggle/input/**/*.*", recursive=True)[:20]) or "Belum ada file di /kaggle/input")

FOTO = None   # contoh: "/kaggle/input/produk-saya/sepatu.jpg"
if FOTO:
    prompt = prompt_natural(
        subjek="the product stays still, sharp and unchanged in the center",
        suasana="clean bright table, soft natural window light, subtle reflections",
        kamera="slow smooth orbit around the product",
        gaya="studio",
    )
    hasil = studio.generate(prompt, image=FOTO, preset="natural")
    display(Video(str(hasil), embed=True, width=270))

## 6. Produksi harian (batch, 2 GPU)
Isi daftar `JOBS`, lalu jalankan. Kalau ada 2 GPU, GPU kedua otomatis ikut membuat video.

Supaya bisa ditinggal (browser boleh ditutup): **Save Version → Save & Run All (Commit)**.
Hasilnya muncul di tab *Output* versi notebook tersebut.

Cara hemat: coba semua prompt dulu dengan `PRESET_HARIAN = "cepat"`, pilih yang bagus, lalu buat ulang dengan
`"natural"` memakai **seed yang sama** (lihat `log.jsonl`) supaya hasilnya mirip tapi lebih tajam.

In [ ]:
PRESET_HARIAN = "natural"
PARALEL = True   # pakai GPU kedua untuk membuat video (kalau ada)

JOBS = [
    prompt_natural("close-up of hands opening a pastel gift box", "confetti falls softly",
                   "bright living room, daylight", "slow motion, handheld", gaya="hp"),
    prompt_natural("a cozy minimalist bedroom at golden hour", "sheer curtains moving in the breeze",
                   kamera="slow pan right", gaya="lifestyle"),
    prompt_natural("fresh orange slices dropping into a glass of sparkling water", "droplets splash in slow motion",
                   "bright summer kitchen counter", "macro close-up", gaya="studio"),
    # foto produk (isi path-nya dulu), boleh pakai preset/seed sendiri:
    # {"prompt": prompt_natural("the product stays unchanged", kamera="slow orbit", gaya="studio"),
    #  "image": "/kaggle/input/produk-saya/tas.jpg"},
    # {"prompt": "...", "preset": "natural_panjang", "seed": 7},
]

studio.encode_semua(JOBS)            # encode semua prompt selagi text encoder masih di GPU
if PARALEL and n_gpu >= 2 and len(studio.workers) == 1:
    studio.text_encoder_to_cpu()     # kosongkan cuda:1 ...
    studio.tambah_worker(muat_worker("cuda:1"))   # ... lalu pakai untuk membuat video
hasil_batch = studio.batch(JOBS, preset=PRESET_HARIAN)

## 7. Unduh semua hasil sekaligus
Folder `mentah/` berisi versi 24 fps sebelum finishing, kalau ingin membandingkan.

In [ ]:
import shutil
zip_path = shutil.make_archive("/kaggle/working/wan_output", "zip", "/kaggle/working/wan_output")
print("Unduh dari panel Output:", zip_path)
print(open("/kaggle/working/wan_output/log.jsonl").read())